## Importações

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

years = range(2019,2025)

graphs = {}
returns = {}

base_modes = "../../data/02_clean"
k = 10

for year in years:
    file_path = f"../../data/04_graphs/synt_{year}_{k}/graph.gpickle"
    with open(file_path, "rb") as f:
        graphs[f"graph_{year}"] = pickle.load(f)
                
        returns[year] = pd.read_parquet(f"../../data/02_clean/synthetic_returns_{year}.parquet")
        returns[year] = np.log(1 + returns[year])

## Funções

In [21]:
import numpy as np
import networkx as nx
from sklearn.decomposition import PCA

def calculate_hybrid_risk_metric(G, use_pca=True, custom_weights=None):
    nodes = list(G.nodes)
    G = G.copy()
    
    for u, v, d in G.edges(data=True):
        # Euclidean distance transformation bounded between 0 and sqrt(2)
        d["distance"] = np.sqrt(2 * max(0.0, 1 - d["weight"]))
    
    # 1. Calculate raw metrics
    degree_dict = dict(nx.degree(G, weight="weight"))
    closeness_dict = nx.closeness_centrality(G, distance="distance")
    eig_dict = nx.eigenvector_centrality_numpy(G, weight="weight")
    
    # Betweenness is calculated but not included in the hybrid metric per your thesis text
    # betweenness_dict = nx.betweenness_centrality(G, weight='distance', normalized=True)
    
    # 2. Standardize metrics using Z-scoring (mean=0, std=1)
    def standardize_dict(d, invert=False):
        vals = np.array(list(d.values()))
        mean_val = vals.mean()
        std_val = vals.std()
        
        if std_val == 0:
            return {k: 0.0 for k in d.keys()}
            
        z_scores = {k: (v - mean_val) / std_val for k, v in d.items()}
        
        if invert:
            z_scores = {k: -v for k, v in z_scores.items()}
            
        return z_scores

    norm_degree = standardize_dict(degree_dict, invert=False)
    norm_closeness = standardize_dict(closeness_dict, invert=False)
    norm_eig = standardize_dict(eig_dict, invert=False)
    
    # 3. Determine Weights
    if use_pca:
        # Build feature matrix (N nodes x 3 metrics)
        features = np.array([
            [norm_degree[n], norm_closeness[n], norm_eig[n]] for n in nodes
        ])
        
        # Extract the First Principal Component
        pca = PCA(n_components=1)
        pca.fit(features)
        pc1_loadings = pca.components_[0]
        
        # PCA vectors can point in arbitrary directions. We take the absolute value
        # to ensure positive loadings, maintaining the logical direction of centrality.
        pc1_loadings = np.abs(pc1_loadings)
        
        # Normalize to sum to 1 (convex combination)
        pca_weights = pc1_loadings / np.sum(pc1_loadings)
        
        weights = {
            'degree': pca_weights[0],
            'closeness': pca_weights[1],
            'eigenvector': pca_weights[2]
        }
    else:
        # Fallback to custom or equal weights if PCA is disabled
        weights = custom_weights if custom_weights else {
            'degree': 1/3, 'closeness': 1/3, 'eigenvector': 1/3
        }

    # 4. Combine metrics with weights
    risk_scores = {}
    for node in nodes:
        score = (
            weights['degree'] * norm_degree[node] +
            weights['closeness'] * norm_closeness[node] +
            weights['eigenvector'] * norm_eig[node] 
        )
        risk_scores[node] = score
    
    return risk_scores, weights

In [22]:
def compute_pozzi_centrality(G, weight_attr="weight"):
    """
    Compute the Pozzi et al. (2013) composite centrality indicators X and Y.

    X aggregates degree + betweenness (both weighted/unweighted).
    Y aggregates eccentricity + closeness + eigenvector (both weighted/unweighted).

    Weighted degree / eigenvector use edge weight  : 1 + R_ij
    Weighted BC / eccentricity / closeness use      : sqrt(2 * (1 - R_ij))  (distance)

    Parameters
    ----------
    G           : nx.Graph with a correlation edge attribute
    weight_attr : str — name of the correlation weight attribute (default "weight")

    Returns
    -------
    pd.DataFrame with columns [Du, Dw, BCu, BCw, Eu, Ew, Cu, Cw, ECu, ECw, X, Y]
    indexed by node.
    """
    N = G.number_of_nodes()

    # --- Build two edge-weighted copies ---
    G_corr = nx.Graph()   # weight = 1 + R  (similarity, for D and EC)
    G_dist = nx.Graph()   # weight = sqrt(2*(1-R))  (distance, for BC/E/C)
    G_corr.add_nodes_from(G.nodes())
    G_dist.add_nodes_from(G.nodes())

    for u, v, data in G.edges(data=True):
        r = data.get(weight_attr, 0.0)
        G_corr.add_edge(u, v, weight=1.0 + r)
        G_dist.add_edge(u, v, weight=np.sqrt(2.0 * (1.0 - r)))

    # --- Unweighted centralities (normalized to [0,1]) ---
    Du  = nx.degree_centrality(G)
    BCu = nx.betweenness_centrality(G, normalized=True)
    Cu  = nx.closeness_centrality(G)
    ECu = nx.eigenvector_centrality_numpy(G)

    # --- Weighted centralities ---
    Dw  = nx.degree_centrality(G_corr)           # similarity weights
    BCw = nx.betweenness_centrality(G_dist, weight="weight", normalized=True)
    Cw  = nx.closeness_centrality(G_dist, distance="weight")
    ECw = nx.eigenvector_centrality_numpy(G_corr, weight="weight")

    # Eccentricity requires connected graph — fall back to largest component
    def _eccentricity(Gh):
        if nx.is_connected(Gh):
            raw = nx.eccentricity(Gh)
        else:
            lcc  = Gh.subgraph(max(nx.connected_components(Gh), key=len))
            raw  = nx.eccentricity(lcc)
            # nodes outside LCC get NaN
            raw  = {n: raw.get(n, np.nan) for n in Gh.nodes()}
        # normalise: invert (higher eccentricity = more peripheral)
        # and scale to [0,1] via min-max
        vals = np.array([v for v in raw.values() if not np.isnan(v)])
        lo, hi = vals.min(), vals.max()
        return {
            n: (raw[n] - lo) / (hi - lo) if not np.isnan(raw.get(n, np.nan)) else np.nan
            for n in Gh.nodes()
        }

    Eu = _eccentricity(G)
    Ew = _eccentricity(G_dist)

    # --- Assemble DataFrame ---
    nodes = list(G.nodes())
    df = pd.DataFrame(index=nodes)

    df["Du"]  = [Du[n]  for n in nodes]
    df["Dw"]  = [Dw[n]  for n in nodes]
    df["BCu"] = [BCu[n] for n in nodes]
    df["BCw"] = [BCw[n] for n in nodes]
    df["Eu"]  = [Eu[n]  for n in nodes]
    df["Ew"]  = [Ew[n]  for n in nodes]
    df["Cu"]  = [Cu[n]  for n in nodes]
    df["Cw"]  = [Cw[n]  for n in nodes]
    df["ECu"] = [ECu[n] for n in nodes]
    df["ECw"] = [ECw[n] for n in nodes]

    # --- Composite scores (each metric already in [0,1]) ---
    df["X"] = (df["Du"] + df["Dw"] + df["BCu"] + df["BCw"]) / 4
    df["Y"] = (df["Eu"] + df["Ew"] + df["Cu"]  + df["Cw"] + df["ECu"] + df["ECw"]) / 6

    df["metric"] = df["X"] + df["Y"]

    return df

## Construção do DataFrame com os nós e as métricas de grafos calculadas para cada modo

In [23]:
for name, G in graphs.items():
    print(name)

graph_2019
graph_2020
graph_2021
graph_2022
graph_2023
graph_2024


In [24]:
data = []

for name, G in graphs.items():
    _, year = name.split("_")

    G = G.copy()
    for u, v, d in G.edges(data=True):
        d["distance"] = np.sqrt(2 * max(0.0, 1 - d["weight"]))

    hrm, _ = calculate_hybrid_risk_metric(G)
    pozzi = compute_pozzi_centrality(G)
    degree_dict = dict(nx.degree(G, weight="weight"))
    closeness_dict = nx.closeness_centrality(G, distance="distance")
    betweenness_dict = nx.betweenness_centrality(G, weight='distance', normalized=True)
    eig_dict = nx.eigenvector_centrality_numpy(G, weight="weight")

    for node in G.nodes():
        data.append({
            "node": node,
            "hrm": hrm[node],
            "pozzi": pozzi.loc[node, "metric"],
            "degree": degree_dict[node],
            "closeness": closeness_dict[node],
            "eig": eig_dict[node],
            "year": year
        })

df = pd.DataFrame(data)

In [25]:
central_portfolios = {}
peripheral_portfolios = {}

q75 = df.groupby("year")["hrm"].quantile(0.80).reset_index(name="p75")
q25 = df.groupby("year")["hrm"].quantile(0.20).reset_index(name="p25")

for year in years:
    p75 = q75.loc[q75["year"] == str(year), "p75"].iloc[0]
    p25 = q25.loc[q25["year"] == str(year), "p25"].iloc[0]

    central_portfolios[year] = df[(df["year"] == str(year)) & (df["hrm"] > p75)]
    peripheral_portfolios[year] = df[(df["year"] == str(year)) & (df["hrm"] < p25)]

In [35]:
centralities = ["central", "peripheral"]

for year in years:
    central_temp = central_portfolios[year]["node"]
    peripheral_temp = peripheral_portfolios[year]["node"]
    returns_temp = pd.read_parquet(
        f"../../data/02_clean/synthetic_returns_{year}.parquet"
    )
    central_port_temp = returns_temp[central_temp]
    peripheral_port_temp = returns_temp[peripheral_temp]

    central_port_temp.to_csv(f"../../data/06_portfolios/synt_central_{year}_{k}.csv")
    peripheral_port_temp.to_csv(f"../../data/06_portfolios/synt_peripheral_{year}_{k}.csv")

## Analysis:

In [36]:
for year in years:
    for k in [1, 10]:
        central_port_temp = pd.read_csv(f"../../data/06_portfolios/synt_central_{year}_{k}.csv")
        peripheral_port_temp = pd.read_csv(f"../../data/06_portfolios/synt_peripheral_{year}_{k}.csv")

        print(f"Year: {year}, k = {k}, Centrality: central")
        columns = set(central_port_temp.columns).union(set(peripheral_port_temp))

        leads = [c for c in columns if "LEAD_" in c]
        lags = [c for c in columns if "LAG_" in c]

        print(f"Number of LEAD columns: {len(leads)}")
        print(f"Number of LAG columns: {len(lags)}")

        print(f"Year: {year}, Centrality: peripheral")
        columns = list(peripheral_port_temp.columns)

        leads = [c for c in columns if "LEAD_" in c]
        lags = [c for c in columns if "LAG_" in c]

        print(f"Number of LEAD columns: {len(leads)}")
        print(f"Number of LAG columns: {len(lags)}")

Year: 2019, k = 1, Centrality: central
Number of LEAD columns: 16
Number of LAG columns: 22
Year: 2019, Centrality: peripheral
Number of LEAD columns: 13
Number of LAG columns: 14
Year: 2019, k = 10, Centrality: central
Number of LEAD columns: 27
Number of LAG columns: 12
Year: 2019, Centrality: peripheral
Number of LEAD columns: 13
Number of LAG columns: 4
Year: 2020, k = 1, Centrality: central
Number of LEAD columns: 16
Number of LAG columns: 22
Year: 2020, Centrality: peripheral
Number of LEAD columns: 13
Number of LAG columns: 14
Year: 2020, k = 10, Centrality: central
Number of LEAD columns: 22
Number of LAG columns: 17
Year: 2020, Centrality: peripheral
Number of LEAD columns: 10
Number of LAG columns: 7
Year: 2021, k = 1, Centrality: central
Number of LEAD columns: 16
Number of LAG columns: 22
Year: 2021, Centrality: peripheral
Number of LEAD columns: 13
Number of LAG columns: 14
Year: 2021, k = 10, Centrality: central
Number of LEAD columns: 23
Number of LAG columns: 19
Year: 2

In [32]:
leads

['LEAD_06',
 'LEAD_47',
 'LEAD_25',
 'LEAD_42',
 'LEAD_01',
 'LEAD_19',
 'LEAD_32',
 'LEAD_37',
 'LEAD_08',
 'LEAD_17',
 'LEAD_09',
 'LEAD_33',
 'LEAD_36',
 'LEAD_10',
 'LEAD_48',
 'LEAD_05']

In [33]:
lags

['LAG_10',
 'LAG_28',
 'LAG_08',
 'LAG_06',
 'LAG_01',
 'LAG_14',
 'LAG_07',
 'LAG_24',
 'LAG_48',
 'LAG_02',
 'LAG_25',
 'LAG_09',
 'LAG_40',
 'LAG_19',
 'LAG_50',
 'LAG_23',
 'LAG_17',
 'LAG_27',
 'LAG_11',
 'LAG_29',
 'LAG_22',
 'LAG_16']